<a href="https://colab.research.google.com/github/InahH-004/Parking-Space-Detection/blob/Jeje/05_full_pipeline_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tahap 5: Inferensi Citra Utuh, Visualisasi Kapasitas & UI

## 0. Setup

In [ ]:
!git clone -b main https://github.com/InahH-004/Parking-Space-Detection.git
%cd Parking-Space-Detection
!pip install -q lightgbm gradio scikit-image

In [ ]:
!mkdir -p models
from google.colab import files
files.upload()
!mv best_model*.pkl models/best_model.pkl
!ls models

In [ ]:
import kagglehub
path = kagglehub.dataset_download("trainingdatapro/parking-space-detection-dataset")
print("Dataset di:", path)

## 1. Fungsi ekstraksi fitur

In [ ]:
import os, cv2, joblib
import numpy as np
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt

CONFIG = {
    "IMG_SIZE": (64, 64),
    "XML_PATH": os.path.join(path, "annotations.xml"),
    "MODEL_PATH": "models/best_model.pkl",
    "LABEL_MAP": {
        "free_parking_space": 0,
        "not_free_parking_space": 1,
        "partially_free_parking_space": 1,
    },
}

In [ ]:
from skimage.feature import hog, local_binary_pattern

FEATURE_CONFIG = {
    "HOG_ORIENTATIONS": 9,
    "HOG_PIXELS_PER_CELL": (8, 8),
    "HOG_CELLS_PER_BLOCK": (2, 2),
    "HOG_BLOCK_NORM": "L2-Hys",
    "LBP_P": 8,
    "LBP_R": 1,
    "LBP_METHOD": "uniform",
    "LBP_GRID": (4, 4),
    "COLOR_BINS": 16
}


def praproses_citra_pcd(img_bgr):
    img_resized = cv2.resize(img_bgr, CONFIG["IMG_SIZE"])
    img_gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img_clahe = clahe.apply(img_gray)
    img_preprocessed = cv2.GaussianBlur(img_clahe, (5, 5), 0)

    img_hsv = cv2.cvtColor(img_resized, cv2.COLOR_BGR2HSV)
    img_lab = cv2.cvtColor(img_resized, cv2.COLOR_BGR2LAB)

    return img_resized, img_preprocessed, img_hsv, img_lab


def ekstrak_fitur_hog(img_gray_prep, visualize=False):
    return hog(
        img_gray_prep,
        orientations=FEATURE_CONFIG["HOG_ORIENTATIONS"],
        pixels_per_cell=FEATURE_CONFIG["HOG_PIXELS_PER_CELL"],
        cells_per_block=FEATURE_CONFIG["HOG_CELLS_PER_BLOCK"],
        block_norm=FEATURE_CONFIG["HOG_BLOCK_NORM"],
        transform_sqrt=True,
        feature_vector=True,
        visualize=visualize
    )


def ekstrak_fitur_lbp(img_gray_prep, return_map=False):
    P = FEATURE_CONFIG["LBP_P"]
    R = FEATURE_CONFIG["LBP_R"]
    method = FEATURE_CONFIG["LBP_METHOD"]
    grid_y, grid_x = FEATURE_CONFIG["LBP_GRID"]
    n_bins = P + 2

    lbp_map = local_binary_pattern(img_gray_prep, P=P, R=R, method=method)

    h, w = lbp_map.shape
    cell_h, cell_w = h // grid_y, w // grid_x
    lbp_hist_list = []

    for gy in range(grid_y):
        for gx in range(grid_x):
            sub_lbp = lbp_map[gy * cell_h:(gy + 1) * cell_h, gx * cell_w:(gx + 1) * cell_w]
            hist, _ = np.histogram(sub_lbp.ravel(), bins=np.arange(0, n_bins + 1), range=(0, n_bins))
            hist = hist.astype(np.float32)
            hist /= (hist.sum() + 1e-7)
            lbp_hist_list.append(hist)

    lbp_features = np.concatenate(lbp_hist_list)
    if return_map:
        return lbp_features, lbp_map
    return lbp_features


def ekstrak_fitur_warna(img_hsv, img_lab):
    bins = FEATURE_CONFIG["COLOR_BINS"]
    color_features = []

    for img_space, ranges in [(img_hsv, [(0, 180), (0, 256), (0, 256)]),
                              (img_lab, [(0, 256), (0, 256), (0, 256)])]:
        for ch_idx in range(3):
            channel = img_space[:, :, ch_idx]
            hist = cv2.calcHist([channel], [0], None, [bins], ranges[ch_idx])
            hist = cv2.normalize(hist, hist).flatten()
            color_features.append(hist)

            max_val = float(ranges[ch_idx][1])
            mean_val = np.mean(channel) / max_val
            std_val = np.std(channel) / max_val
            color_features.append(np.array([mean_val, std_val], dtype=np.float32))

    return np.concatenate(color_features)


def ekstrak_vektor_fitur_pcd(img_bgr):
    _, img_prep, img_hsv, img_lab = praproses_citra_pcd(img_bgr)
    feat_hog = ekstrak_fitur_hog(img_prep, visualize=False).astype(np.float32)
    feat_lbp = ekstrak_fitur_lbp(img_prep, return_map=False).astype(np.float32)
    feat_color = ekstrak_fitur_warna(img_hsv, img_lab).astype(np.float32)

    vektor_gabungan = np.concatenate([feat_hog, feat_lbp, feat_color])
    return vektor_gabungan, (feat_hog, feat_lbp, feat_color)

In [ ]:
COLOR_FREE = (0, 255, 0)      # BGR
COLOR_OCCUPIED = (0, 0, 255)

assert os.path.exists(CONFIG["MODEL_PATH"]), "models/best_model.pkl tidak ditemukan"
model = joblib.load(CONFIG["MODEL_PATH"])
print("Model dimuat:", type(model.named_steps["clf"]).__name__)

_dummy = np.random.randint(0, 255, (50, 40, 3), dtype=np.uint8)
print("Dimensi vektor fitur:", ekstrak_vektor_fitur_pcd(_dummy)[0].shape)

## 2. Baca koordinat slot dari XML

In [ ]:
def parse_slots(image_node):
    slots = []
    for poly in image_node.findall("polygon"):
        pts = np.array([[int(float(a)), int(float(b))]
                        for a, b in (p.split(",") for p in poly.get("points").split(";"))], dtype=np.int32)
        x, y, w, h = cv2.boundingRect(pts)
        slots.append({
            "polygon": pts,
            "bbox": (x, y, x + w, y + h),
            "label_asli": CONFIG["LABEL_MAP"].get(poly.get("label"), 1),
        })
    return slots

def get_image_node(img_name):
    root = ET.parse(CONFIG["XML_PATH"]).getroot()
    for node in root.findall("image"):
        if node.get("name") == img_name:
            return node
    raise ValueError(f"{img_name} tidak ada di annotations.xml")

all_names = [n.get("name") for n in ET.parse(CONFIG["XML_PATH"]).getroot().findall("image")]
print(len(all_names), "citra, contoh:", all_names[:3])

## 3. Prediksi semua slot, gambar kotak, dan hitung kapasitas

In [ ]:
def prediksi_slot(img_bgr, slots):
    H, W = img_bgr.shape[:2]
    fitur = []
    for s in slots:
        x1, y1, x2, y2 = s["bbox"]
        x1, y1, x2, y2 = max(0, x1), max(0, y1), min(W, x2), min(H, y2)
        patch = img_bgr[y1:y2, x1:x2]
        vec, _ = ekstrak_vektor_fitur_pcd(patch)
        fitur.append(vec)
    return model.predict(np.array(fitur))

def gambar_overlay(img_bgr, slots, preds):
    out = img_bgr.copy()
    for s, p in zip(slots, preds):
        warna = COLOR_OCCUPIED if p == 1 else COLOR_FREE
        cv2.polylines(out, [s["polygon"].reshape(-1, 1, 2)], True, warna, 3)
    return out

def hitung_kapasitas(preds):
    total = len(preds)
    kosong = int(np.sum(preds == 0))
    return {"total": total, "kosong": kosong, "terisi": total - kosong,
            "persen_kosong": 100 * kosong / total if total else 0.0}

def tambah_panel_ringkasan(img_bgr, k):
    teks = f"Kosong: {k['kosong']}/{k['total']}  ({k['persen_kosong']:.1f}%)"
    out = img_bgr.copy()
    cv2.rectangle(out, (0, 0), (out.shape[1], 70), (0, 0, 0), -1)
    cv2.putText(out, teks, (20, 48), cv2.FONT_HERSHEY_SIMPLEX, 1.4, (255, 255, 255), 3)
    return out

def proses_citra(img_name):
    img = cv2.imread(os.path.join(path, img_name))
    slots = parse_slots(get_image_node(img_name))
    preds = prediksi_slot(img, slots)
    k = hitung_kapasitas(preds)
    hasil = tambah_panel_ringkasan(gambar_overlay(img, slots, preds), k)
    acc = np.mean(preds == np.array([s["label_asli"] for s in slots]))
    return cv2.cvtColor(hasil, cv2.COLOR_BGR2RGB), k, acc

## 4. Uji pada satu citra

In [ ]:
hasil_rgb, k, acc = proses_citra(all_names[0])
plt.figure(figsize=(12, 8)); plt.imshow(hasil_rgb); plt.axis("off")
plt.title("Hijau = kosong | Merah = terisi"); plt.show()
print(f"Total slot : {k['total']}")
print(f"Slot kosong: {k['kosong']}  |  Terisi: {k['terisi']}")
print(f"Ketersediaan: {k['persen_kosong']:.1f}%")
print(f"Kecocokan dengan label XML: {acc*100:.2f}%")

## 5. Evaluasi ke seluruh citra


In [ ]:
for n in all_names:
    _, k, acc = proses_citra(n)
    print(f"{n:15s} slot={k['total']:3d}  kosong={k['kosong']:3d} ({k['persen_kosong']:5.1f}%)  cocok XML={acc*100:5.1f}%")

## 6. Demo UI (Gradio)

In [ ]:
import gradio as gr

def demo(img_name):
    hasil_rgb, k, acc = proses_citra(img_name)
    ringkasan = (f"Total slot: {k['total']}\n"
                 f"Slot kosong: {k['kosong']}\n"
                 f"Slot terisi: {k['terisi']}\n"
                 f"Ketersediaan: {k['persen_kosong']:.1f}%")
    return hasil_rgb, ringkasan

ui = gr.Interface(
    fn=demo,
    inputs=gr.Dropdown(all_names, value=all_names[0], label="Pilih citra tempat parkir"),
    outputs=[gr.Image(label="Hasil deteksi (hijau=kosong, merah=terisi)"),
             gr.Textbox(label="Ringkasan kapasitas", lines=4)],
    title="Deteksi Kapasitas Lahan Parkir",
    description="HOG + LBP + Warna + LightGBM (tanpa deep learning)",
)
ui.launch(share=True)